# 02 Hypothesis Testing

This notebook is executed and includes outputs plus interpretation notes.

## Goal
Test whether monthly patterns, OSB exposure, and sector/severity associations are statistically meaningful.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
sys.path.append(str(Path.cwd().parent))
clean = pd.read_excel('../data/processed/kmo_incidents_clean.xlsx')
istanbul = pd.read_excel('../data/processed/istanbul_enriched.xlsx')
with open('../reports/hypothesis_results.json', encoding='utf-8') as f:
    results = json.load(f)
print(json.dumps(results, indent=2, ensure_ascii=False))

{
  "H1_seasonality_kruskal": {
    "H": 11.810444267496177,
    "p_value": 0.3780589571696683,
    "epsilon_squared": 0.009883466676782643
  },
  "H2_osb_exposure_mannwhitney": {
    "U": 12171.5,
    "p_value": 9.913055982834709e-31,
    "osb_province_median": 10.0,
    "non_osb_province_median": 1.0
  },
  "H3_osb_exposure_correlations": {
    "osb_parcels_spearman": 0.7947635916056632,
    "osb_parcels_p_value": 7.821841143164724e-17,
    "osb_area_spearman": 0.7493686668718624,
    "osb_area_p_value": 3.6782180553461e-14
  },
  "H4_sector_severity_chi_square": {
    "chi2": 90.32395452962544,
    "p_value": 6.506124205918993e-11,
    "dof": 20
  },
  "H4_province_osb_exposure_severity_chi_square": {
    "chi2": 4.508137026316152,
    "p_value": 0.10497127757416845,
    "dof": 2
  }
}


In [2]:
monthly = clean.groupby(['year','month']).size().reset_index(name='count')
monthly.pivot(index='month', columns='year', values='count').fillna(0).astype(int)

year,2017,2018,2019,2020,2021,2022,2023,2024
month,,,,,,,,
1,11,38,42,48,27,35,31,61
2,8,33,44,26,22,42,41,45
3,0,28,54,37,30,41,36,40
4,0,36,34,29,26,35,27,50
5,12,29,58,37,28,44,36,69
6,19,29,53,32,37,65,50,66
7,15,46,52,42,49,55,71,79
8,6,39,43,39,31,64,62,75
9,23,40,49,61,45,55,49,33


### H1 Interpretation
The Kruskal-Wallis seasonality test is not significant after adding 2024. This means the current evidence does not support a strong month-level seasonal difference in reported industrial incidents across years. The temporal story is weaker than the spatial/exposure story.

In [3]:
province_counts = clean.groupby('il').agg(
    incidents=('Tarih','count'),
    osb_parcels=('osb_parcels','first'),
    osb_area_hectare=('osb_area_hectare','first'),
    has_osb=('has_city_osb_exposure','first')
).sort_values('incidents', ascending=False)
province_counts.head(20)

,incidents,osb_parcels,osb_area_hectare,has_osb
il,,,,
İstanbul,993,1863,2417.60,True
İzmir,399,3487,5070.78,True
Bursa,254,2688,5723.07,True
Kocaeli,223,2129,4032.00,True
Tekirdağ,187,2170,5633.82,True
Ankara,114,14155,9173.46,True
Sakarya,87,1119,2678.70,True
Denizli,74,510,1347.95,True
Adana,72,927,4119.04,True


### H2/H3 Interpretation
The OSB exposure tests are strong: provinces with OSB exposure have higher incident counts, and incident counts correlate strongly with OSB parcel and area measures. This does not mean OSBs cause fires; it means industrial capacity is a major exposure denominator and must be controlled for.

In [4]:
sector_severity = pd.crosstab(clean['sektor_std'], clean['severity'])
sector_severity

severity,high,low,medium
sektor_std,,,
"Ağaç,Kağıt,Mobilya",21,752,45
Bilinmeyen,9,662,20
Boya,4,39,7
Enerji,5,56,4
Gıda,14,309,18
"Kauçuk,Plastik",7,359,16
"Kozmetik,Temizlik",2,40,5
Metal,41,505,54
"Petrokimya,Yağ",11,111,13


### H4 Interpretation
Sector and severity are statistically associated. This is substantively useful because sectors differ in materials, processes, equipment, and storage conditions. However, it should be interpreted carefully because unknown/missing causes and reporting bias remain important limitations.